# Phase 1-5 / 1-6 — TIAGE topic-shift detection + 종합 Gate

> **사전 조건**: 이 노트북 실행 전에 반드시 `setup_colab.ipynb` 셀 0~7을 먼저 실행해야 합니다 (Hi-EM repo, 벤치마크 레포 4종 clone, 패키지 설치, 데이터 다운로드, bge 모델 로드).

Hi-EM segmenter의 chit-chat 대화(PersonaChat 기반)에서 topic 경계 감지 성능 확인. TopiOCQA(factoid)와 독립 benchmark.

- **데이터**: `benchmarks/tiage/data/personachat/anno/test/anno_test.json` — 100 conv / 1564 turns / 315 shifts (Cohen's Kappa 0.48)
- **실행 스크립트**: `scripts/run_tiage_segmentation.py`
- **Gate 기준** (plan.md Phase 1-4 준용): Hi-EM F1 > cosine baseline AND F1 > 0.4 AND 턴당 overhead ≤ 200ms

## 0. 환경 감지 + 프로젝트 루트

In [16]:
import os, subprocess

try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    PROJECT_DIR = '/content/Hi-EM'
else:
    try:
        PROJECT_DIR = subprocess.check_output(
            ['git', 'rev-parse', '--show-toplevel'], text=True
        ).strip()
    except Exception:
        PROJECT_DIR = os.path.abspath('..')

# 사전 조건: setup_colab.ipynb를 먼저 실행한 상태여야 함
assert os.path.exists(PROJECT_DIR), (
    f'{PROJECT_DIR} 없음 — setup_colab.ipynb를 먼저 실행하세요.\n'
    'Colab: setup_colab.ipynb 셀 0~7 순서대로 실행 후 이 노트북 재실행.'
)

os.chdir(PROJECT_DIR)
print(f"env: {'Colab' if IS_COLAB else 'local'}")
print(f"cwd: {os.getcwd()}")

env: Colab
cwd: /content/Hi-EM


## 1. 사전 조건 검증
`setup_colab.ipynb`가 이미 실행됐는지 확인. Hi-EM repo / TIAGE 벤치마크 / 패키지 모두 준비된 상태여야 함.

준비 안 됐으면 이 셀이 명확한 에러로 실패하니, `setup_colab.ipynb` 셀 0~7 먼저 실행 후 이 노트북 재실행.

In [17]:
import os, sys

# 사전 조건 체크 — 부족하면 setup_colab.ipynb 실행 안내
prereq = [
    ('Hi-EM repo', os.path.exists(PROJECT_DIR)),
    ('TIAGE benchmark',
     os.path.exists(os.path.join(PROJECT_DIR, 'benchmarks', 'tiage'))),
    ('TIAGE test split',
     os.path.exists(os.path.join(
         PROJECT_DIR, 'benchmarks', 'tiage',
         'data', 'personachat', 'anno', 'test', 'anno_test.json'))),
]

missing = [name for name, ok in prereq if not ok]
if missing:
    raise RuntimeError(
        f'사전 조건 미충족: {missing}.\n'
        f'→ setup_colab.ipynb 셀 0~7 먼저 실행하세요. '
        f'(cell 3가 Hi-EM + 벤치마크 레포 4종 자동 clone)'
    )
print('[ok] prereq satisfied (Hi-EM + TIAGE)')

# 파이썬 의존성 — setup_colab cell 4가 설치한 것 검증만
try:
    import sentence_transformers, transformers, numpy  # noqa
    print(f'[deps] sentence-transformers={sentence_transformers.__version__}, '
          f'transformers={transformers.__version__}')
except ImportError as e:
    raise RuntimeError(
        f'필수 패키지 누락 ({e}). setup_colab.ipynb 셀 4 (패키지 설치) 먼저 실행.'
    )

import torch
print(f'[torch] CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  device: {torch.cuda.get_device_name(0)}')

[ok] prereq satisfied (Hi-EM + TIAGE)
[deps] sentence-transformers=5.4.0, transformers=5.0.0
[torch] CUDA: True
  device: NVIDIA A100-SXM4-80GB


## 2. TIAGE test split 평가 실행
A100 기준 ~5-15초 소요.

In [18]:
import subprocess, sys, time

t0 = time.perf_counter()
r = subprocess.run(
    [sys.executable, 'scripts/run_tiage_segmentation.py', '--split', 'test'],
    capture_output=True, text=True,
)
print(f"[took {time.perf_counter() - t0:.1f}s, returncode={r.returncode}]\n")
print(r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr)

[took 16.8s, returncode=0]

[data] loading TIAGE test...
  100 conv / 1564 turns / 315 shifts
[encode] bge-base-en-v1.5...
  encoded 1564 turns in 0.9s (0.59 ms/turn) on cuda
(a) all-boundary : P=0.215 R=1.000 F1=0.354  V=0.597 ARI=0.000
(b) cosine θ=0.525: P=0.332 R=0.575 F1=0.421  V=0.650 ARI=0.384
(c) Hi-EM α=1.0, λ=10.0, σ₀²=0.01: P=0.245 R=0.451 F1=0.317  V=0.624 ARI=0.317  (assign 0.115 ms/turn)
(c') Hi-EM freq-shift: P=0.239 R=0.895 F1=0.377  V=0.615 ARI=0.143

report → /content/Hi-EM/outputs/phase-1-tiage.md



## 3. 결과 markdown 표시

In [19]:
from pathlib import Path
from IPython.display import Markdown, display

md = Path('outputs/phase-1-tiage.md')
if md.exists():
    display(Markdown(md.read_text()))
else:
    print(f"{md} missing — cell 2 먼저 실행")

# Phase 1-3 (augment) — TIAGE topic-shift detection

실행: `python scripts/run_tiage_segmentation.py --split test` (device=cuda)

## 데이터 — TIAGE test
- dialogs: 100
- total turns: 1564
- topic-shift labels ('1'): 315 (shift rate 0.215 / transition)

## 지표

### Topic-shift F1 (turn-transition binary) + Clustering quality (V-measure / ARI, per-dialog 평균)

| Method | Precision | Recall | F1 | V-measure | ARI |
|---|---|---|---|---|---|
| (a) all-boundary | 0.215 | 1.000 | 0.354 | 0.597 | 0.000 |
| (b) cosine-threshold (θ=0.525) | 0.332 | 0.575 | 0.421 | 0.650 | 0.384 |
| (c) Hi-EM persistence (α=1.0, λ=10.0, σ₀²=0.01) | 0.245 | 0.451 | 0.317 | 0.624 | 0.317 |
| (c') Hi-EM freq-shift (α=10, λ=1, σ₀²=0.1) | 0.239 | 0.895 | 0.377 | 0.615 | 0.143 |

**지표 의미**:
- **F1**: turn-transition 단위 boundary 정확도 (binary classification)
- **V-measure**: homogeneity × completeness 조화평균. 0~1, 1=완벽한 클러스터 일치
- **ARI** (Adjusted Rand Index): chance-corrected pairwise agreement. 1=완벽, 0=random, 음수 가능

## Latency
- embed: 0.9s / 0.59 ms/turn
- Hi-EM assign: 179.6 ms total / 0.115 ms/turn
- 총 overhead: 0.70 ms/turn

## Gate 판정 (plan.md Phase 1-4 criteria, TIAGE 적용)

- Hi-EM F1 > cosine baseline F1: **False** (0.317 vs 0.421)
- Hi-EM F1 > 0.4: **False** (0.317)
- 턴당 overhead ≤ 200ms (LLM 1000ms의 20%): **True** (0.70 ms)

**Gate**: FAIL


## 4. TIAGE Gate 판정 (Phase 1-5)

In [20]:
from pathlib import Path

tiage_md = Path('outputs/phase-1-tiage.md').read_text()
if '**Gate**: PASS' in tiage_md:
    tiage_pass = True
    verdict = '✓ TIAGE Gate PASS'
elif '**Gate**: FAIL' in tiage_md:
    tiage_pass = False
    verdict = '✗ TIAGE Gate FAIL'
else:
    tiage_pass = None
    verdict = '? Gate 표기 못 찾음 — md 수동 확인'

print(verdict)
print()
print('→ Phase 1-6 종합 판정은 다음 셀에서 TopiOCQA 결과와 합산')

✗ TIAGE Gate FAIL

→ Phase 1-6 종합 판정은 다음 셀에서 TopiOCQA 결과와 합산


## 5. Phase 1-6 종합 Gate (TopiOCQA + TIAGE 둘 다 PASS 여야 Phase 2 진입)

In [21]:
from pathlib import Path

topi_path = Path('outputs/phase-1-topiocqa.md')
if topi_path.exists():
    topi_md = topi_path.read_text()
    topi_pass = '**Gate 결과: PASS' in topi_md
else:
    topi_pass = None

print('Phase 1-6 종합 Gate')
print(f"  TopiOCQA (Phase 1-4): {'PASS' if topi_pass else ('FAIL' if topi_pass is False else 'N/A - 결과 파일 없음')}")
print(f"  TIAGE    (Phase 1-5): {'PASS' if tiage_pass else ('FAIL' if tiage_pass is False else 'N/A')}")
print()
if topi_pass and tiage_pass:
    print('→ PHASE 1 CLEARED. Phase 2 (LTM + Memory window) 진입 자격 확보.')
elif topi_pass is False or tiage_pass is False:
    print('→ 하나 이상 FAIL. `context/06-decision-log.md`에 기록 후:')
    print('  1. HP regime 재검토 (benchmark별 다른 HP가 맞는가)')
    print('  2. 옵션 A 구조적 한계 확인 시 옵션 D(multi-signal) escalation')
else:
    print('→ 결과 파일 부족. Phase 1-3 (TopiOCQA) + Phase 1-5 (TIAGE) 모두 완료 필요.')

Phase 1-6 종합 Gate
  TopiOCQA (Phase 1-4): PASS
  TIAGE    (Phase 1-5): FAIL

→ 하나 이상 FAIL. `context/06-decision-log.md`에 기록 후:
  1. HP regime 재검토 (benchmark별 다른 HP가 맞는가)
  2. 옵션 A 구조적 한계 확인 시 옵션 D(multi-signal) escalation


In [22]:
import subprocess

# 1. Colab의 Hi-EM이 어떤 브랜치/commit인지
r = subprocess.run(
    ['git', '-C', '/content/Hi-EM', 'log', '--oneline', '-3'],
    capture_output=True,
    text=True
)
print("Colab git log -3:")
print(r.stdout)

r = subprocess.run(
    ['git', '-C', '/content/Hi-EM', 'branch', '--show-current'],
    capture_output=True,
    text=True
)
print("Colab branch:", r.stdout.strip())

# 2. scripts/run_tiage_segmentation.py에 V-measure/ARI 코드가 있는지
r = subprocess.run(
    ['grep', '-c', 'v_measure_score', '/content/Hi-EM/scripts/run_tiage_segmentation.py'],
    capture_output=True,
    text=True
)
print(f"v_measure_score 등장 횟수: {r.stdout.strip()} (0이면 옛 버전)")

Colab git log -3:
bf65f20 feat(scripts): add V-measure and ARI clustering metrics + branch decision log
ab656fa feat(scripts): add V-measure and ARI clustering metrics
a59cd14 docs(policy): notebook-script split + cascade check rule + .ipynb tracking

Colab branch: feat/sem-v2
v_measure_score 등장 횟수: 2 (0이면 옛 버전)
